# 🧩 Eche — Keras implementation and reproducibility
**Gradient Descent Workshop · Group 1 · Echezona (Eche) Oji · 9078881**

This standalone contribution completes Challenges C1/C2 and reflections C12/C13. The supplied root notebook is preserved byte-for-byte. Data are the four workshop observations, $y=2x+1$; no downloads or credentials are required to run the experiments.

Run all cells with the Python environment installed from the root `requirements.txt`. Seeds, initial weights, full-batch size, learning rate and epoch counts are explicit. All errors below are training errors on four points, not evidence of generalization.

In [1]:
# 🧩 Eche — Echezona (Eche) Oji, 9078881
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import sys, platform, json
from pathlib import Path
from importlib.metadata import version
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

keras.utils.set_random_seed(8245)
tf.config.experimental.enable_op_determinism()
pio.renderers.default = 'plotly_mimetype'
ARTIFACTS = Path('eche_results')
ARTIFACTS.mkdir(exist_ok=True)
x = tf.constant([[1.], [2.], [3.], [4.]], dtype=tf.float32)
y = 2*x + 1
print('Python:', sys.version.split()[0], '| Platform:', platform.platform(), platform.machine())
print({p: version(p) for p in ['numpy', 'tensorflow', 'keras', 'plotly']})
print('x:', x.numpy().ravel(), 'y:', y.numpy().ravel())

# A fixed full batch gives exactly one optimizer update per epoch.
options = tf.data.Options()
options.threading.private_threadpool_size = 1
options.threading.max_intra_op_parallelism = 1
train_data = tf.data.Dataset.from_tensor_slices((x, y)).batch(4).with_options(options)

def linear_model():
    return keras.Sequential([
        layers.Input(shape=(1,)),
        layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    ])

def mse(model):
    return float(tf.reduce_mean(tf.square(model(x, training=False)-y)).numpy())

def show_curve(histories, title, filename):
    fig = go.Figure()
    for label, values in histories.items():
        fig.add_trace(go.Scatter(x=list(range(1,len(values)+1)), y=values,
                                 mode='lines', name=label))
    fig.update_layout(title=title, xaxis_title='Epoch', yaxis_title='MSE (log scale)',
                      yaxis_type='log', template='plotly_white', hovermode='x unified')
    fig.write_html(ARTIFACTS/filename, include_plotlyjs=True, full_html=True)
    fig.show()
    return fig


Python: 3.12.10 | Platform: macOS-27.0.1-arm64-arm-64bit arm64
{'numpy': '2.5.5', 'tensorflow': '2.21.0', 'keras': '3.15.1', 'plotly': '7.2.0'}
x: [1. 2. 3. 4.] y: [3. 5. 7. 9.]


## 🧩 Eche — C1: `compile()` and `fit()`

The model has one weight and one bias, with an explicit `Input(shape=(1,))`. Zero initialization is appropriate for this single linear neuron (it is not a suitable general initialization for symmetric hidden units). Every experiment starts at $w=b=0$. `compile()` chooses SGD and MSE; `fit()` performs the epoch/batch loops, differentiation, updates and loss recording.

For this full-batch setup, `history.history['loss']` records the loss before that epoch's update. We separately recompute MSE using the final weights, avoiding an off-by-one comparison. We compare 200 epochs at 0.01, 2,000 at 0.01, and 200 at 0.05.

In [2]:
# 🧩 Eche — Echezona (Eche) Oji, 9078881
c1_models, c1_histories, c1_results = {}, {}, {}
for label, lr, epochs in [('C1 baseline', .01, 200), ('C1 longer', .01, 2000),
                           ('C1 faster', .05, 200)]:
    model = linear_model()
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=lr), loss='mse', jit_compile=False)
    history = model.fit(train_data, epochs=epochs, verbose=0)
    c1_models[label] = model
    c1_histories[label] = [float(v) for v in history.history['loss']]
    weight, bias = (float(a.ravel()[0]) for a in model.get_weights())
    c1_results[label] = dict(lr=lr, epochs=epochs, weight=weight, bias=bias,
        weight_error=abs(weight-2), bias_error=abs(bias-1),
        last_recorded_loss=c1_histories[label][-1], final_mse=mse(model))
    print(label, json.dumps(c1_results[label], indent=2))
show_curve(c1_histories, 'C1: epoch budget and learning rate', 'c1_loss.html')


/Users/apple/Documents/Codex/2026-11-24/wor/work/gd_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


C1 baseline {
  "lr": 0.01,
  "epochs": 200,
  "weight": 2.0535154342651367,
  "bias": 0.8426593542098999,
  "weight_error": 0.05351543426513672,
  "bias_error": 0.1573406457901001,
  "last_recorded_loss": 0.0041594430804252625,
  "final_mse": 0.0041345576755702496
}
C1 longer {
  "lr": 0.01,
  "epochs": 2000,
  "weight": 2.000243902206421,
  "bias": 0.9992846250534058,
  "weight_error": 0.00024390220642089844,
  "bias_error": 0.0007153749465942383,
  "last_recorded_loss": 8.611579005446401e-08,
  "final_mse": 8.554491159884492e-08
}
C1 faster {
  "lr": 0.05,
  "epochs": 200,
  "weight": 2.0047738552093506,
  "bias": 0.9859648942947388,
  "weight_error": 0.004773855209350586,
  "bias_error": 0.01403510570526123,
  "last_recorded_loss": 3.390514393686317e-05,
  "final_mse": 3.289844244136475e-05
}


## 🧩 Eche — C2: custom `GradientTape` loop

This keeps the workshop architecture: one input, eight ReLU hidden units and one linear output. A tape records the differentiable forward pass and MSE; `tape.gradient` calculates a gradient for every trainable weight and bias. SGD applies the updates. Unlike C1, we explicitly store both pre-update loss and post-update full-data MSE.

We run the requested 200 epochs first, then continue the same model to 1,000 epochs to examine convergence. Individual hidden-layer weights need not equal 2 or 1: those values describe the complete input–output relationship, not each parameter of a multi-layer network.

In [3]:
# 🧩 Eche — Echezona (Eche) Oji, 9078881
def train_custom(model, epochs, lr=.01):
    optimizer = keras.optimizers.SGD(learning_rate=lr)
    loss_fn = keras.losses.MeanSquaredError()
    pre_losses, post_losses = [], []
    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            predictions = model(x, training=True)
            loss = loss_fn(y, predictions)
        grads = tape.gradient(loss, model.trainable_weights)
        assert all(g is not None for g in grads), 'Disconnected trainable parameter'
        for g in grads:
            tf.debugging.assert_all_finite(g, 'Non-finite gradient')
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        pre_losses.append(float(loss.numpy()))
        post_losses.append(mse(model))
    return pre_losses, post_losses

keras.utils.set_random_seed(8245)
c2_model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(8, activation='relu', kernel_initializer=keras.initializers.GlorotUniform(seed=8245)),
    layers.Dense(1, kernel_initializer=keras.initializers.GlorotUniform(seed=8246))
])
c2_pre, c2_post = train_custom(c2_model, 200)
c2_at_200 = mse(c2_model)
print('C2 after 200 epochs, post-update MSE:', c2_at_200)
extra_pre, extra_post = train_custom(c2_model, 800)
# Recreating plain SGD is equivalent here: no momentum or schedule state is used.
c2_pre += extra_pre
c2_post += extra_post
print('C2 after 1000 epochs, post-update MSE:', mse(c2_model))
print(' x     true y     predicted y     error')
c2_predictions = c2_model(x, training=False).numpy().ravel()
for xv, target, pred in zip(x.numpy().ravel(), y.numpy().ravel(), c2_predictions):
    print(f'{xv:2.0f} {target:10.6f} {pred:15.6f} {pred-target:11.6f}')
show_curve({'C2 pre-update': c2_pre, 'C2 post-update': c2_post},
           'C2: custom-loop loss each epoch', 'c2_loss.html')


C2 after 200 epochs, post-update MSE: 0.0011873251060023904
C2 after 1000 epochs, post-update MSE: 3.881268639815971e-10
 x     true y     predicted y     error
 1   3.000000        2.999969   -0.000031
 2   5.000000        4.999985   -0.000015
 3   7.000000        7.000001    0.000001
 4   9.000000        9.000018    0.000018


## 🧩 Eche — Comparing the interfaces fairly

| Responsibility | C1: `fit()` | C2: custom loop |
|---|---|---|
| Data batching | Consumes our prebatched dataset; can batch array inputs | We supply the complete four-point tensor each step |
| Epoch iteration | Built in | Explicit Python loop |
| Forward pass / differentiation | Built-in training step | Model call inside `GradientTape`, then `tape.gradient` |
| Parameter update | Configured optimizer called internally | Explicit `apply_gradients` |
| Loss history | Returns a `History` object | Append losses ourselves |
| Callbacks / validation | Standard interfaces supplied | Must implement or invoke explicitly |

C1 is linear and C2 has a hidden layer, so their loss curves alone cannot isolate an API effect. The next check repeats C1's linear model with identical initialization, learning rate, full batches and 200 updates through the custom loop. Their weights and pre-update losses should agree within floating-point tolerance.

In [4]:
# 🧩 Eche — Echezona (Eche) Oji, 9078881
matched_model = linear_model()
matched_pre, matched_post = train_custom(matched_model, 200, .01)
for fit_weight, custom_weight in zip(c1_models['C1 baseline'].get_weights(), matched_model.get_weights()):
    np.testing.assert_allclose(fit_weight, custom_weight, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(c1_histories['C1 baseline'], matched_pre, rtol=2e-5, atol=1e-6)
assert c1_results['C1 longer']['final_mse'] < c1_results['C1 baseline']['final_mse']
assert c1_results['C1 longer']['weight_error'] < .01
assert c1_results['C1 longer']['bias_error'] < .01
assert mse(c2_model) < c2_pre[0]
assert np.all(np.isfinite(c2_predictions))
print('PASS: matched-loop weights/losses agree; longer C1 training approaches w=2, b=1; C2 loss falls.')
show_curve({'fit: linear baseline': c1_histories['C1 baseline'],
            'custom: same linear model': matched_pre},
           'Controlled comparison: same model and updates', 'matched_loop_loss.html')
results = {'c1': c1_results, 'c2': {'mse_200': c2_at_200, 'mse_1000': mse(c2_model),
           'predictions': c2_predictions.tolist()}, 'matched_loop_check': 'passed'}
(ARTIFACTS/'results.json').write_text(json.dumps(results, indent=2))


PASS: matched-loop weights/losses agree; longer C1 training approaches w=2, b=1; C2 loss falls.


1210

## 🧩 Eche — Reflections C12 and C13

**C12. In Keras, what do you gain from `model.fit(...)`, and what do you lose compared with a custom training loop?**

`fit()` removes repetitive training code and provides epoch iteration, optimizer updates, metrics/history, callbacks and validation support. For array inputs it can also handle batching and shuffling; here we supplied batching ourselves through a dataset. The tradeoff is that the default step hides the exact gradient computation and update order, making unusual training algorithms harder to express through configuration alone. A custom loop gives direct access to gradients and intermediate values, but we must manage logging, validation and stopping. Overriding `train_step()` is a middle ground that keeps `fit()` conveniences.

**C13. Which framework felt most transparent for learning? Which felt most convenient?**

*Suggested reflection to review and personalize:* The explicit `GradientTape` loop is the most transparent in this contribution because I can point to the forward pass, loss, gradients and update individually. The supplied PyTorch loop exposes the same sequence through `backward()` and `step()`. Keras `fit()` is the most convenient for standard training because a compile/fit call replaces the repeated loop. The controlled linear experiment shows that this convenience changes how the work is expressed, not the gradient-descent rule. This comparison is based on the code structure and the experiments here; it is not a claim about personal framework preference or training speed.

## 🧩 Eche — Insights grounded in these runs


In [5]:
# 🧩 Eche — Echezona (Eche) Oji, 9078881
base, longer, faster = (c1_results[k] for k in ['C1 baseline', 'C1 longer', 'C1 faster'])
display(Markdown(f"""
1. **Epoch budget matters.** At learning rate 0.01, C1's MSE falls from **{base['final_mse']:.6g}** after 200 updates to **{longer['final_mse']:.6g}** after 2,000. Its bias moves from **{base['bias']:.6f}** to **{longer['bias']:.6f}**, toward the true value 1. A small prediction error does not mean every parameter has converged.
2. **Learning rate affects progress per update.** At the same 200-update budget, changing the learning rate from 0.01 to 0.05 changes final MSE from **{base['final_mse']:.6g}** to **{faster['final_mse']:.6g}**. This observation applies to this tiny linear problem, not every network or larger learning rate.
3. **Architecture and API are separate choices.** C2 reaches MSE **{c2_at_200:.6g}** at 200 epochs and **{mse(c2_model):.6g}** at 1,000, but uses a different architecture from C1. The separate matching-model check passes, demonstrating agreement between `fit()` and the explicit loop when the optimization conditions match. None of these four-point training results measures generalization.
"""))



1. **Epoch budget matters.** At learning rate 0.01, C1's MSE falls from **0.00413456** after 200 updates to **8.55449e-08** after 2,000. Its bias moves from **0.842659** to **0.999285**, toward the true value 1. A small prediction error does not mean every parameter has converged.
2. **Learning rate affects progress per update.** At the same 200-update budget, changing the learning rate from 0.01 to 0.05 changes final MSE from **0.00413456** to **3.28984e-05**. This observation applies to this tiny linear problem, not every network or larger learning rate.
3. **Architecture and API are separate choices.** C2 reaches MSE **0.00118733** at 200 epochs and **3.88127e-10** at 1,000, but uses a different architecture from C1. The separate matching-model check passes, demonstrating agreement between `fit()` and the explicit loop when the optimization conditions match. None of these four-point training results measures generalization.


## 🧩 Eche — Reproducibility and integration

- Fresh-environment verification details are in `../REPRODUCIBILITY.txt`.
- Root `requirements.txt` pins the six requested direct dependencies; `requirements-lock.txt` captures the full tested environment, including transitive dependencies.
- The root original notebook, supplied README and John's notebook remain unchanged. Merge these contribution cells into Section 16 and add C12/C13 to Section 17; avoid overwriting teammates' global `x`, `y` or `model` variables without rerunning dependent cells.
- A final merged notebook was not supplied. Its fresh-kernel verification remains pending until Sultan provides the merged artifact.

Sources used to verify the API explanations: [Keras model training APIs](https://keras.io/api/models/model_training_apis/) and [Keras custom TensorFlow training loop guide](https://keras.io/guides/writing_a_custom_training_loop_in_tensorflow/).